In [4]:
from google.colab import drive
import os

# Ensure the mount point is empty before mounting
if os.path.exists('/content/drive'):
    os.system('rm -rf /content/drive/*')

drive.mount('/content/drive', force_remount=True)

import glob, numpy as np

BASE = '/content/drive/MyDrive/deepfake_v2'

real_list = sorted(glob.glob(f'{BASE}/roop_input_v2/*.mp4'))
fake_list = sorted(glob.glob(f'{BASE}/fake_full_v2/*.mp4'))

print(f"Real videos : {len(real_list)}")
print(f"Fake videos : {len(fake_list)}")
print(f"\nFirst 3 real: {[v.split('/')[-1] for v in real_list[:3]]}")
print(f"First 3 fake: {[v.split('/')[-1] for v in fake_list[:3]]}")

Mounted at /content/drive
Real videos : 92
Fake videos : 92

First 3 real: ['sinhala_001.mp4', 'sinhala_002.mp4', 'sinhala_003.mp4']
First 3 fake: ['sinhala_001_fake.mp4', 'sinhala_002_fake.mp4', 'sinhala_003_fake.mp4']


In [5]:
import os, cv2, glob
import numpy as np
from sklearn.model_selection import train_test_split

BASE     = '/content/drive/MyDrive/deepfake_v2'
REAL_DIR = f'{BASE}/roop_input_v2'
FAKE_DIR = f'{BASE}/fake_full_v2'
SAVE_DIR = f'{BASE}/deepfake_processed_v3'
os.makedirs(SAVE_DIR, exist_ok=True)

# Keep 15 frames — matches your original training
# Changing to 30 needs much more GPU memory
FRAMES   = 15
IMG_SIZE = 224

detector = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    'haarcascade_frontalface_default.xml'
)

def extract_faces(path, n=15):
    cap   = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < n:
        cap.release()
        return None

    indices = np.linspace(0, total-1, n, dtype=int)
    faces   = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        dets = detector.detectMultiScale(
            gray, scaleFactor=1.1,
            minNeighbors=4, minSize=(50, 50)
        )

        if len(dets) > 0:
            dets = sorted(dets,
                key=lambda x: x[2]*x[3], reverse=True)
            x, y, w, h = dets[0]
            face = frame[y:y+h, x:x+w]
            face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))
            face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
            faces.append(face.astype(np.float32) / 255.0)
        else:
            faces.append(
                np.zeros((IMG_SIZE, IMG_SIZE, 3), np.float32)
            )

    cap.release()

    if len(faces) < n:
        return None

    return np.array(faces[:n])

# ── Process REAL videos ────────────────────────────────
seqs, labels = [], []

real_list = sorted(glob.glob(f'{REAL_DIR}/*.mp4'))
print(f"Processing {len(real_list)} REAL videos...")

for i, p in enumerate(real_list):
    s = extract_faces(p, FRAMES)
    if s is not None:
        seqs.append(s)
        labels.append(0)     # 0 = REAL
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{len(real_list)} done")

print(f"Real extracted: {sum(l==0 for l in labels)}")

# ── Process FAKE videos ────────────────────────────────
fake_list = sorted(glob.glob(f'{FAKE_DIR}/*.mp4'))
print(f"\nProcessing {len(fake_list)} FAKE videos...")

for i, p in enumerate(fake_list):
    s = extract_faces(p, FRAMES)
    if s is not None:
        seqs.append(s)
        labels.append(1)     # 1 = FAKE face-swap
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{len(fake_list)} done")

print(f"Fake extracted: {sum(l==1 for l in labels)}")

# ── Build and verify arrays ────────────────────────────
X = np.array(seqs,   dtype=np.float32)
y = np.array(labels, dtype=np.int32)

print(f"\nX shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"Real    : {sum(y==0)}")
print(f"Fake    : {sum(y==1)}")
print(f"y unique: {np.unique(y)}")

# VERIFY — must show both 0 and 1
if len(np.unique(y)) < 2:
    print("ERROR — labels are missing one class!")
else:
    print("Labels OK — both classes present")

Processing 92 REAL videos...
  10/92 done
  20/92 done
  30/92 done
  40/92 done
  50/92 done
  60/92 done
  70/92 done
  80/92 done
  90/92 done
Real extracted: 92

Processing 92 FAKE videos...
  10/92 done
  20/92 done
  30/92 done
  40/92 done
  50/92 done
  60/92 done
  70/92 done
  80/92 done
  90/92 done
Fake extracted: 92

X shape : (184, 15, 224, 224, 3)
y shape : (184,)
Real    : 92
Fake    : 92
y unique: [0 1]
Labels OK — both classes present


In [6]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.15,
    stratify=y,
    random_state=42
)
X_tr, X_va, y_tr, y_va = train_test_split(
    X_tr, y_tr,
    test_size=0.15,
    stratify=y_tr,
    random_state=42
)

print(f"Train: {X_tr.shape}  Real={sum(y_tr==0)} Fake={sum(y_tr==1)}")
print(f"Val  : {X_va.shape}  Real={sum(y_va==0)} Fake={sum(y_va==1)}")
print(f"Test : {X_te.shape}  Real={sum(y_te==0)} Fake={sum(y_te==1)}")

np.save(f'{SAVE_DIR}/X_train_seq.npy', X_tr)
np.save(f'{SAVE_DIR}/X_val_seq.npy',   X_va)
np.save(f'{SAVE_DIR}/X_test_seq.npy',  X_te)
np.save(f'{SAVE_DIR}/y_train_seq.npy', y_tr)
np.save(f'{SAVE_DIR}/y_val_seq.npy',   y_va)
np.save(f'{SAVE_DIR}/y_test_seq.npy',  y_te)

# Verify sizes are correct
for name, arr in [('y_train', y_tr), ('y_val', y_va), ('y_test', y_te)]:
    kb = arr.nbytes / 1024
    print(f"{name}: shape={arr.shape}  size={kb:.1f}KB  "
          f"unique={np.unique(arr, return_counts=True)}")

Train: (132, 15, 224, 224, 3)  Real=66 Fake=66
Val  : (24, 15, 224, 224, 3)  Real=12 Fake=12
Test : (28, 15, 224, 224, 3)  Real=14 Fake=14
y_train: shape=(132,)  size=0.5KB  unique=(array([0, 1], dtype=int32), array([66, 66]))
y_val: shape=(24,)  size=0.1KB  unique=(array([0, 1], dtype=int32), array([12, 12]))
y_test: shape=(28,)  size=0.1KB  unique=(array([0, 1], dtype=int32), array([14, 14]))


In [7]:
import tensorflow as tf
import numpy as np
import os

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import (
    Input, Dense, Bidirectional, GRU,
    GlobalAveragePooling2D, TimeDistributed,
    Dropout, Conv2D
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    f1_score, classification_report, confusion_matrix
)

DATA = '/content/drive/MyDrive/deepfake_v2/deepfake_processed_v3'
SAVE = '/content/drive/MyDrive/deepfake_v2/deepfake_models'
os.makedirs(SAVE, exist_ok=True)

X_train = np.load(f'{DATA}/X_train_seq.npy')
X_val   = np.load(f'{DATA}/X_val_seq.npy')
X_test  = np.load(f'{DATA}/X_test_seq.npy')
y_train = np.load(f'{DATA}/y_train_seq.npy')
y_val   = np.load(f'{DATA}/y_val_seq.npy')
y_test  = np.load(f'{DATA}/y_test_seq.npy')

print("Train:", X_train.shape,
      "Real:", sum(y_train==0), "Fake:", sum(y_train==1))
print("Val  :", X_val.shape)
print("Test :", X_test.shape)

# ── AUGMENTATION ──────────────────────────────────────
def augment(seq, lbl):
    seq = tf.image.random_brightness(seq, max_delta=0.15)
    seq = tf.cond(
        tf.random.uniform(()) > 0.5,
        lambda: tf.image.flip_left_right(seq),
        lambda: seq
    )
    seq = tf.clip_by_value(seq, 0.0, 1.0)
    return seq, lbl

def maybe_augment(seq, lbl):
    return tf.cond(
        tf.random.uniform(()) > 0.5,
        lambda: augment(seq, lbl),
        lambda: (seq, lbl)
    )

train_ds = (
    tf.data.Dataset
    .from_tensor_slices((X_train, y_train))
    .shuffle(len(X_train))
    .map(maybe_augment, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(4)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    tf.data.Dataset
    .from_tensor_slices((X_val, y_val))
    .batch(4)
    .prefetch(tf.data.AUTOTUNE)
)

# ── CBAM LAYERS ───────────────────────────────────────
@tf.keras.utils.register_keras_serializable()
class ChannelAttention(tf.keras.layers.Layer):
    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio
    def build(self, s):
        ch = s[-1]
        self.fc1 = Dense(ch // self.ratio, activation='relu')
        self.fc2 = Dense(ch, activation='sigmoid')
    def call(self, x):
        a = tf.reduce_mean(x, axis=[1,2], keepdims=True)
        m = tf.reduce_max(x,  axis=[1,2], keepdims=True)
        return x * (self.fc2(self.fc1(a)) +
                    self.fc2(self.fc1(m)))
    def get_config(self):
        c = super().get_config()
        c['ratio'] = self.ratio
        return c

@tf.keras.utils.register_keras_serializable()
class SpatialAttention(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    def build(self, s):
        self.conv = Conv2D(1, 7, padding='same',
                           activation='sigmoid')
    def call(self, x):
        a = tf.reduce_mean(x, axis=-1, keepdims=True)
        m = tf.reduce_max(x,  axis=-1, keepdims=True)
        return x * self.conv(tf.concat([a, m], axis=-1))

# ── BUILD MODEL ───────────────────────────────────────
def build_model():
    base = MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    # Unfreeze last 50 layers
    for layer in base.layers[:-50]:
        layer.trainable = False
    for layer in base.layers[-50:]:
        layer.trainable = True

    fi = Input(shape=(224, 224, 3))
    x  = base(fi, training=False)
    x  = ChannelAttention()(x)
    x  = SpatialAttention()(x)
    x  = GlobalAveragePooling2D()(x)
    frame_model = Model(fi, x, name='frame_extractor')

    inp = Input(shape=(15, 224, 224, 3), name='seq_input')
    f   = TimeDistributed(frame_model)(inp)
    x   = Bidirectional(GRU(64, return_sequences=True))(f)
    x   = Bidirectional(GRU(32, return_sequences=False))(x)
    x   = Dense(64, activation='relu')(x)
    x   = Dropout(0.4)(x)
    out = Dense(2, activation='softmax')(x)

    return Model(inp, out, name='Condition_C_v2')

model = build_model()

# KEY FIX — use 1e-4 not 1e-5
# 1e-5 is too slow and gets stuck — needs 50+ epochs
# 1e-4 converges properly in 15-25 epochs
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

callbacks = [
    EarlyStopping(
        patience=7,
        restore_best_weights=True,
        monitor='val_accuracy'
    ),
    ReduceLROnPlateau(
        factor=0.5, patience=3,
        min_lr=1e-7,
        monitor='val_loss'
    ),
    ModelCheckpoint(
        f'{SAVE}/condition_c_v2_best.keras',
        save_best_only=True,
        monitor='val_accuracy'
    )
]

print("\nTraining...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=callbacks
)

model.save(f'{SAVE}/condition_c_v2_final.keras')
print("Saved.")

Train: (132, 15, 224, 224, 3) Real: 66 Fake: 66
Val  : (24, 15, 224, 224, 3)
Test : (28, 15, 224, 224, 3)
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "Condition_C_v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ seq_input (InputLayer)          │ (None, 15, 224, 224,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 15, 1280)       │     2,669,123 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 15, 128)        │       516,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        31,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,221,381 (12.29 MB)

 Trainable params: 2,818,501 (10.75 MB)

 Non-trainable params: 402,880 (1.54 MB)


Training...
Epoch 1/40
33/33 ━━━━━━━━━━━━━━━━━━━━ 130s 979ms/step - accuracy: 0.5606 - loss: 0.7413 - val_accuracy: 0.5417 - val_loss: 0.6989 - learning_rate: 1.0000e-04
Epoch 2/40
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 286ms/step - accuracy: 0.5682 - loss: 0.7012 - val_accuracy: 0.5417 - val_loss: 0.6847 - learning_rate: 1.0000e-04
Epoch 3/40
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 311ms/step - accuracy: 0.6364 - loss: 0.6643 - val_accuracy: 0.6250 - val_loss: 0.6498 - learning_rate: 1.0000e-04
Epoch 4/40
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 307ms/step - accuracy: 0.5909 - loss: 0.6238 - val_accuracy: 0.6667 - val_loss: 0.6510 - learning_rate: 1.0000e-04
Epoch 5/40
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 313ms/step - accuracy: 0.7500 - loss: 0.5143 - val_accuracy: 0.7083 - val_loss: 0.6109 - learning_rate: 1.0000e-04
Epoch 6/40
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 311ms/step - accuracy: 0.8030 - loss: 0.4117 - val_accuracy: 0.8333 - val_loss: 0.4673 - learning_rate: 1.0000e-04
Epoch 7/40
33/33 ━━━━━━━━━━━━━━━━━━━━ 9s 282

In [8]:
y_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_fake = 1.0 - y_prob[:, 0]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score((y_test > 0).astype(int), y_fake)
f1  = f1_score((y_test > 0).astype(int),
               (y_pred > 0).astype(int))

print("=" * 50)
print("CONDITION C_v2 FINAL RESULTS")
print("=" * 50)
print(f"Accuracy : {acc:.4f}  (original was 0.6923)")
print(f"AUC-ROC  : {auc:.4f}  (original was 0.7743)")
print(f"F1       : {f1:.4f}  (original was 0.7857)")
print()
print(classification_report(
    y_test, y_pred,
    target_names=['Real', 'Fake']
))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

CONDITION C_v2 FINAL RESULTS
Accuracy : 0.8214  (original was 0.6923)
AUC-ROC  : 0.9235  (original was 0.7743)
F1       : 0.8000  (original was 0.7857)

              precision    recall  f1-score   support

        Real       0.76      0.93      0.84        14
        Fake       0.91      0.71      0.80        14

    accuracy                           0.82        28
   macro avg       0.84      0.82      0.82        28
weighted avg       0.84      0.82      0.82        28

Confusion Matrix:
[[13  1]
 [ 4 10]]
